# Session 2 — Your turn: The transformer block

Count the parameters of a block from its shapes, and check the property that makes rotary position embeddings work: the score between a rotated query and key depends only on how far apart they are.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np
from aij_check import check, close, summary

## Task 1: Count a block's parameters

For a pre-norm block with model width `d`, `h` heads (no change in total size) and an MLP hidden width `4 * d`, write `block_params(d)` that returns a dictionary with:
`attention` (the combined query/key/value matrix `d x 3d` plus its bias `3d`, and the output matrix `d x d` plus its bias `d`), `mlp` (two matrices `d x 4d` and `4d x d` with their biases) and `norms` (two layer norms, each with a scale and a shift of size `d`), and `total`.

In [ ]:
def block_params(d):
    ...

In [ ]:
gpt2 = block_params(768)
check("GPT-2's attention has 2,362,368 parameters", lambda: gpt2["attention"] == 2_362_368)
check("GPT-2's MLP has 4,722,432 parameters", lambda: gpt2["mlp"] == 4_722_432)
check("the MLP has about twice the parameters of the attention", lambda: 1.9 < gpt2["mlp"] / gpt2["attention"] < 2.1)
check("twelve such blocks hold about 85 million parameters", lambda: 84e6 < 12 * gpt2["total"] < 86e6)
check("the total is the sum of the parts", lambda: gpt2["total"] == gpt2["attention"] + gpt2["mlp"] + gpt2["norms"])
summary()

## Task 2: Rotary embeddings in two dimensions

RoPE rotates a vector by an angle proportional to its position. Write `rotate(v, position, theta=0.3)` that rotates the 2D vector `v` by the angle `position * theta` (counter-clockwise). Then, for the fixed vectors `q` and `k` below,
compute `score(m, n) = rotate(q, m) @ rotate(k, n)` and store in `same_gap` whether `score(3, 1)`, `score(10, 8)` and `score(7, 5)` (all a gap of 2) are equal, and in `different_gap` whether `score(3, 1)` differs from `score(3, 0)`.

In [ ]:
q, k = np.array([1.0, 0.5]), np.array([0.3, 1.0])


def rotate(v, position, theta=0.3):
    ...


def score(m, n):
    return rotate(q, m) @ rotate(k, n)


same_gap = ...
different_gap = ...

In [ ]:
check("rotating by position 0 changes nothing", lambda: close(rotate(np.array([1.0, 2.0]), 0), [1.0, 2.0], 1e-12))
check("rotation keeps the length of a vector", lambda: close(np.linalg.norm(rotate(q, 5)), np.linalg.norm(q), 1e-12))
check("equal gaps give equal scores", lambda: same_gap is True)
check("different gaps give different scores", lambda: different_gap is True)
summary()